In [476]:
import numpy as np
import pymap3d as pm
from rotation_matrix import rotation_matrix
np.set_printoptions(suppress=True)

In [477]:
# Observer
a1_lat = 36.7341597
# a1_lon = -117.2166654
a1_lon = -117.2165206
# a1_alt = 11000 * 0.3048
a1_alt = 11500 * 0.3048
a1_roll = 0
a1_pitch = 0
a1_heading = 0
# E (wing), N (nose), U (tail)
radar_position = [0, 475, 0]

In [478]:
# Target
rap_lat = 36.7290516
# rap_lat = 36.7341590
rap_lon = -117.2165206
rap_alt = 11500 * 0.3048
rap_roll = 0
rap_pitch = 0
rap_heading = 0

In [479]:
# rap_lats = [36.7290516, 36.7290516]
# rap_lons = [-117.2165206, -117.2165206]
# rap_alts = [11500 * 0.3048, 11500 * 0.3048]
# rap_rolls = 0
# rap_pitchs = 5
# rap_headings = 0

In [480]:
# Convert geodetic of target and observer to ENU between observer and target
enu_a12rap = pm.geodetic2enu(rap_lat, rap_lon, rap_alt, a1_lat, a1_lon, a1_alt)
# enu_a12rap = pm.geodetic2enu(a1_lat, a1_lon, a1_alt, rap_lat, rap_lon, rap_alt)
enu_a12rap = np.reshape(enu_a12rap, (-1, 1))
print(enu_a12rap)

[[   0.        ]
 [-567.17173439]
 [  -0.02528257]]


In [481]:
# Account for radar position in observer body coordinate system
radar_translation = np.reshape(radar_position, (-1, 1)) / 12 * 0.3048 # meters
print(radar_translation)

[[ 0.   ]
 [12.065]
 [ 0.   ]]


In [482]:
# Account for observer rotation about body coordinate system
a1_rmat = rotation_matrix(a1_roll, a1_pitch, a1_heading, isTarget=True)
print(a1_rmat)

[[-1.  0.  0.]
 [-0. -1.  0.]
 [ 0.  0.  1.]]


In [483]:
# Combine rotation and translation
# [3x3][3x1]
radar_xyz_vector = a1_rmat @ radar_translation
print(radar_xyz_vector)

[[  0.   ]
 [-12.065]
 [  0.   ]]


In [484]:
# Combine observer ECEF coordinates with radar vector
radar_enu = enu_a12rap - radar_xyz_vector
print(radar_enu)

[[   0.        ]
 [-555.10673439]
 [  -0.02528257]]


In [485]:
# radar_lat, radar_lon, radar_alt = pm.ecef2geodetic(radar_ecef[0], radar_ecef[1], radar_ecef[2])
# print('A1', a1_lat, a1_lon, a1_alt)
# print('Radar', radar_lat, radar_lon, radar_alt)
# print('Raptor', rap_lat, rap_lon, rap_alt)

In [486]:
# Convert coordinates to ENU frame relative to the observers location
# az, el, slant_range = pm.geodetic2aer(rap_lat, rap_lon, rap_alt, radar_lat, radar_lon, radar_alt)
az, el, slant_range = np.squeeze(pm.enu2aer(radar_enu[0], radar_enu[1], radar_enu[2]))
# az, el, slant_range = pm.geodetic2aer(rap_lats, rap_lons, rap_alts, a1_lat, a1_lon, a1_alt)
print('Azimuth: ', az)
print('Elevation: ', el)
print('Slant range (ft): ', slant_range/0.3048)
print('Ground range (ft): ', np.sqrt(slant_range**2 - np.abs(a1_alt - rap_alt)**2))

Azimuth:  180.0
Elevation:  -0.0026095608162220773
Slant range (ft):  1821.2163220671575
Ground range (ft):  555.1067349660697


In [487]:
az = np.radians(az)
az_rmat = np.array([
    [np.cos(az), -1*np.sin(az), 0],
    [np.sin(az), np.cos(az), 0],
    [0, 0, 1]
])
el = np.radians(el)
el_rmat = np.array([
    [np.cos(el), 0, np.sin(el)],
    [0, 1, 0],
    [-1*np.sin(el), 0, np.cos(el)]
])

In [488]:
# a1_rmat = rotation_matrix(a1_roll, a1_pitch, a1_heading, isTarget=False)
rap_rmat = rotation_matrix(rap_roll, rap_pitch, rap_heading, isTarget=True)

In [489]:
rmat = rap_rmat @ az_rmat @ el_rmat

In [490]:
look = np.atan2(-1*rmat[1, 0], -1*rmat[0, 0])
look = np.mod(np.round(np.degrees(look), 6), 360)

depr = np.asin(rmat[2, 0])
depr = np.degrees(depr)

twist = np.atan2(-1*rmat[2, 1], rmat[2, 2])
twist = np.degrees(twist)

print(f'Look: {look:.2f}')
print(f'Depression: {depr:.2f}')
print(f'Twist: {twist:.2f}')

Look: 0.00
Depression: 0.00
Twist: -0.00
